# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shaheerkhan1117/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The rule, in plain words:** priority = how much a page's traffic is declining, weighted by how
much traffic is actually at stake, discounted whenever the evidence behind the trend is thin or
noisy. No row gets forced into an action its own data can't support — if there aren't enough
observed days to trust a front-half/back-half split, the action is `review`, not `declining` or
`growing`, no matter what the raw percentage says.

**Inputs** — all built in `w03_feature_leakage_check`, all knowable from `month = '2026-03'`
alone, no columns beyond `fact_content_daily_performance`:
- `pct_change` — back-half vs front-half impressions within the month (same definition as
  `w03_data_contract`'s label, used here as a rule input, not a prediction target).
- `total_impressions` — how much traffic is riding on this page.
- `active_days` — how many of the 31 days actually had impressions (confidence in the split).
- `position_volatility` — day-to-day noise in ranking position (confidence in the trend itself).

**Reason codes** (a row can carry more than one):
- `DECLINE_20PLUS` — back-half impressions down 20%+ vs front-half.
- `GROWTH_20PLUS` — back-half impressions up 20%+ vs front-half.
- `STABLE_TREND` — neither of the above.
- `SPARSE_DATA` — fewer than 10 active days this month; too little of the month observed to
  trust any front/back split. Overrides the trend-based action outright.
- `HIGH_VOLATILITY` — `position_volatility` in the top 10% among pages with enough days to
  compute it; ranking is too noisy to read a clean signal from position alone.
- `LOW_TRAFFIC` — `total_impressions` in the bottom quartile; small swings on a tiny base look
  like big percentages and usually aren't real.

**Action assignment:** `SPARSE_DATA` forces `review` regardless of the trend. Otherwise
`DECLINE_20PLUS` → `declining`, `GROWTH_20PLUS` → `growing`, else `stable`. `HIGH_VOLATILITY`
and `LOW_TRAFFIC` never change the action bucket — they discount the *priority score* instead,
so a noisy or tiny-volume decline still gets flagged, just ranked lower than a clean one.

In [6]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/shaheerkhan1117/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import getpass
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb, numpy as np, pandas as pd
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":    f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":    f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

MONTH = "2026-03"  # same mid-panel month as w03_data_contract and w03_feature_leakage_check

Paste your Hugging Face READ token (hf_...): ··········


## 2. Build the ranked queue (writes the CSV)

Rebuilds the feature set from `w03_feature_leakage_check` — `fact_content_daily_performance`
only, `month = '2026-03'` only, no `dim_content` join (no product flags in scope for this
rule) — then scores, ranks, and writes `work/outputs/baseline_action_score.csv`.

In [7]:
raw = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)                                            AS total_impressions,
        SUM(gsc_clicks)                                                 AS total_clicks,
        AVG(gsc_avg_position)                                           AS avg_position,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0)  AS active_days,
        STDDEV_SAMP(gsc_avg_position)                                   AS position_volatility,
        SUM(CASE WHEN report_date < DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) AS imp_first_half,
        SUM(CASE WHEN report_date >= DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) AS imp_second_half
    FROM {TABLES['fact_daily']}
    WHERE month = '{MONTH}'
    GROUP BY 1, 2
    HAVING SUM(CASE WHEN report_date < DATE '{MONTH}-16' THEN gsc_impressions ELSE 0 END) > 0
""").df()

df = raw.copy()
df["pct_change"] = (df["imp_second_half"] - df["imp_first_half"]) / df["imp_first_half"]
df["volatility_is_filled"] = df["position_volatility"].isna().astype(int)
df["position_volatility"] = df["position_volatility"].fillna(0.0)

# Thresholds — computed from this slice's own distribution, not hardcoded guesses.
volatility_p90 = df.loc[df["volatility_is_filled"] == 0, "position_volatility"].quantile(0.90)
traffic_p25 = df["total_impressions"].quantile(0.25)

def reason_codes(row):
    codes = []
    if row["active_days"] < 10:
        codes.append("SPARSE_DATA")
    if row["pct_change"] <= -0.20:
        codes.append("DECLINE_20PLUS")
    elif row["pct_change"] >= 0.20:
        codes.append("GROWTH_20PLUS")
    else:
        codes.append("STABLE_TREND")
    if row["volatility_is_filled"] == 0 and row["position_volatility"] >= volatility_p90:
        codes.append("HIGH_VOLATILITY")
    if row["total_impressions"] <= traffic_p25:
        codes.append("LOW_TRAFFIC")
    return codes

df["reason_codes"] = df.apply(reason_codes, axis=1)

def assign_action(codes):
    if "SPARSE_DATA" in codes:
        return "review"
    if "DECLINE_20PLUS" in codes:
        return "declining"
    if "GROWTH_20PLUS" in codes:
        return "growing"
    return "stable"

df["action"] = df["reason_codes"].apply(assign_action)

def confidence_note(row):
    if "SPARSE_DATA" in row["reason_codes"]:
        return f"low — only {row['active_days']} active days observed this month"
    note = f"based on {row['active_days']} active days"
    if "HIGH_VOLATILITY" in row["reason_codes"]:
        note += ", but ranking position is unusually noisy this month"
    if "LOW_TRAFFIC" in row["reason_codes"]:
        note += ", on a small traffic base"
    return note

df["confidence_note"] = df.apply(confidence_note, axis=1)

# Priority score: only meaningful for the declining bucket, discounted when confidence is thin.
def priority_score(row):
    if row["action"] != "declining":
        return 0.0
    weight = 1.0
    if "HIGH_VOLATILITY" in row["reason_codes"]:
        weight *= 0.5
    if "LOW_TRAFFIC" in row["reason_codes"]:
        weight *= 0.25
    return round(-row["pct_change"] * np.log1p(row["total_impressions"]) * weight, 4)

df["priority_score"] = df.apply(priority_score, axis=1)
df["reason_codes_str"] = df["reason_codes"].apply(",".join)

ranked = df.sort_values("priority_score", ascending=False).reset_index(drop=True)
ranked.insert(0, "rank", ranked.index + 1)

out_cols = ["rank", "client_hash_id", "content_hash_id", "action", "reason_codes_str",
            "confidence_note", "priority_score", "pct_change", "total_impressions",
            "active_days", "avg_position", "position_volatility"]

os.makedirs("work/outputs", exist_ok=True)
ranked[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"{len(ranked):,} rows scored and ranked, written to work/outputs/baseline_action_score.csv")
ranked["action"].value_counts()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

151,981 rows scored and ranked, written to work/outputs/baseline_action_score.csv


,count
action,
growing,58640
declining,33858
stable,33176
review,26307


## 3. Top-20 review

For each of the top 20 by `priority_score`: the action, its reason codes, a confidence note
built from the same evidence the action used, and a `what_would_make_it_wrong` caveat generated
from the specific reason codes each row carries — not a generic disclaimer repeated 20 times.
After running this in Colab, read the actual 20 rows and add any row-specific exceptions the
templated caveat misses (e.g. a page you happen to know just changed URL).

In [8]:
CAVEAT_BY_CODE = {
    "DECLINE_20PLUS": "wrong if the drop is a site-wide dip (indexing issue, algorithm update) "
                       "rather than something specific to this page — worth checking whether "
                       "other pages for the same client fell the same week before refreshing this one.",
    "HIGH_VOLATILITY": "wrong if this page's position genuinely swings this much every month — "
                        "a flagged-volatile page can be a normal noisy performer, not a real decline.",
    "LOW_TRAFFIC": "wrong if this is a new or low-priority page still ramping — a big percentage "
                   "swing on a tiny base is closer to noise than signal.",
    "SPARSE_DATA": "wrong in the other direction too — a page under review here might already be "
                   "declining or growing, there's just not enough of the month observed to say so yet.",
}

def what_would_make_it_wrong(codes):
    notes = [CAVEAT_BY_CODE[c] for c in codes if c in CAVEAT_BY_CODE]
    return " ".join(notes) if notes else "no specific caveat beyond the confidence note above."

top20 = ranked.head(20).copy()
top20["what_would_make_it_wrong"] = top20["reason_codes"].apply(what_would_make_it_wrong)

review_cols = ["rank", "content_hash_id", "action", "reason_codes_str", "confidence_note",
               "what_would_make_it_wrong"]
pd.set_option("display.max_colwidth", None)
top20[review_cols]

,rank,content_hash_id,action,reason_codes_str,confidence_note,what_would_make_it_wrong
0,1,content_9c057b66c30a3abb,declining,DECLINE_20PLUS,based on 31 active days,"wrong if the drop is a site-wide dip (indexing issue, algorithm update) rather than something specific to this page — worth checking whether other pages for the same client fell the same week before refreshing this one."
1,2,content_afa44be39cea94ca,declining,DECLINE_20PLUS,based on 31 active days,"wrong if the drop is a site-wide dip (indexing issue, algorithm update) rather than something specific to this page — worth checking whether other pages for the same client fell the same week before refreshing this one."
2,3,content_65c75874a23fca87,declining,DECLINE_20PLUS,based on 31 active days,"wrong if the drop is a site-wide dip (indexing issue, algorithm update) rather than something specific to this page — worth checking whether other pages for the same client fell the same week before refreshing this one."
3,4,content_62673eea26c31c17,declining,DECLINE_20PLUS,based on 31 active days,"wrong if the drop is a site-wide dip (indexing issue, algorithm update) rather than something specific to this page — worth checking whether other pages for the same client fell the same week before refreshing this one."
4,5,content_945d6ff91386c817,declining,DECLINE_20PLUS,based on 31 active days,"wrong if the drop is a site-wide dip (indexing issue, algorithm update) rather than something specific to this page — worth checking whether other pages for the same client fell the same week before refreshing this one."
5,6,content_6a56a2183dc01691,declining,DECLINE_20PLUS,based on 29 active days,"wrong if the drop is a site-wide dip (indexing issue, algorithm update) rather than something specific to this page — worth checking whether other pages for the same client fell the same week before refreshing this one."
6,7,content_8abf2671c081e29e,declining,DECLINE_20PLUS,based on 31 active days,"wrong if the drop is a site-wide dip (indexing issue, algorithm update) rather than something specific to this page — worth checking whether other pages for the same client fell the same week before refreshing this one."
7,8,content_f97d175377d97f04,declining,DECLINE_20PLUS,based on 20 active days,"wrong if the drop is a site-wide dip (indexing issue, algorithm update) rather than something specific to this page — worth checking whether other pages for the same client fell the same week before refreshing this one."
8,9,content_9bcfb1e373c01b7a,declining,DECLINE_20PLUS,based on 31 active days,"wrong if the drop is a site-wide dip (indexing issue, algorithm update) rather than something specific to this page — worth checking whether other pages for the same client fell the same week before refreshing this one."
9,10,content_9e0a8a913953b8d3,declining,DECLINE_20PLUS,based on 29 active days,"wrong if the drop is a site-wide dip (indexing issue, algorithm update) rather than something specific to this page — worth checking whether other pages for the same client fell the same week before refreshing this one."


## 4. Weak picks + leakage check

**Weak picks:** rows that rank in the top 20 despite carrying `HIGH_VOLATILITY` or
`LOW_TRAFFIC` — the discount weights should already push these down, so any that still surface
near the top are worth a manual look before shipping the queue as-is.

**Leakage check:** confirm every input to the rule sits inside `month = '2026-03'` (no future
window past the sealed panel edge), and confirm no `dim_content` columns — product flags
included — were ever joined into the scoring table.

In [9]:
# Weak picks: discounted reason codes that still made the top 20
weak_picks = top20[top20["reason_codes_str"].str.contains("HIGH_VOLATILITY|LOW_TRAFFIC")]
print(f"Weak picks in the top 20: {len(weak_picks)}")
weak_picks[["rank", "content_hash_id", "reason_codes_str", "priority_score"]]

Weak picks in the top 20: 0


,rank,content_hash_id,reason_codes_str,priority_score


In [12]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/shaheerkhan1117/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import getpass
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb, numpy as np, pandas as pd
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":    f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":    f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

MONTH = "2026-03"  # same mid-panel month as w03_data_contract and w03_feature_leakage_check

# Leakage check 1 — every date behind pct_change sits inside this month, nothing past the panel edge
date_bounds = con.sql(f"""
    SELECT MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {TABLES['fact_daily']}
    WHERE month = '{MONTH}'
""").df()
print("Dates behind this queue:")
print(date_bounds)
assert date_bounds['max_date'][0].strftime('%Y-%m-%d') <= f"{MONTH}-31", "found a date past this month's end"

# Leakage check 2 — no dim_content columns, product flags included, ever entered the scoring frame
scoring_columns = set(raw.columns) | set(df.columns)
dim_content_cols_full = set(con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']} LIMIT 0").df()['column_name'])

# Exclude common identifier columns that are also present in fact_daily and essential for scoring
excluded_from_leakage_check = {'client_hash_id', 'content_hash_id'}
dim_content_cols_for_leakage = dim_content_cols_full - excluded_from_leakage_check

overlap = scoring_columns & dim_content_cols_for_leakage
print(f"\ndim_content columns present in the scoring table (excluding common IDs): {overlap or 'none'}")
assert not overlap, "a dim_content column (other than common IDs) leaked into the scoring table"

print("\nBoth checks pass: nothing here reaches past the observed month, nothing from dim_content was joined.")

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dates behind this queue:
    min_date   max_date
0 2026-03-01 2026-03-31

dim_content columns present in the scoring table (excluding common IDs): none

Both checks pass: nothing here reaches past the observed month, nothing from dim_content was joined.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.